# Phase 7 — Hyperparameter sweep

Phase 6 flagged that the decile-1 lift of 1.93× was operationally useful but well short of industry-best. The cheapest thing to try next is *not* feature engineering — it's tuning the model we already have. Phase 4 picked XGBoost defaults by hand; this phase runs a proper search.

**Design choices (locked in `src/tune.py`):**

1. **Random search over 15 trials** (the first is Phase 4's defaults, so we can rank our previous guess against the search). The search space has 576 grid points — full grid would need 1,152 fits, ~5 hours. Random sampling at 15 trials is enough to identify the *direction* of improvement.
2. **Time-series CV with 2 folds.** Same principle as everywhere else in the project: never split randomly across time when production scores future loans.
3. **25% time-ordered subsample of the base training slice (2007-2015).** Per-fit cost drops from ~80s to ~10-15s. Hyperparameters tune model *capacity*, not data quirks — the best params on a subsample transfer back.
4. **Optimize ROC-AUC.** PR-AUC scales with base rate; default rate climbs year-over-year in the data, so PR-AUC across folds isn't directly comparable. ROC-AUC is.

**Outputs:**
- `outputs/tune_results.csv` — full sweep results, one row per trial.
- `outputs/models/xgb_v3_tuned.joblib` — best params refit on full base + isotonic calibration on 2016.

Re-running the sweep takes ~10 minutes; the cells below default to *reading* the saved CSV. Use `python src/_smoke_phase7.py` to regenerate.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT = Path(r'C:\Users\User\Desktop\Credit Risk Prediction System')
sys.path.insert(0, str(PROJECT / 'src'))

from prepare import load_processed
from models import split_xy, evaluate, model_path
from evaluate import time_calibration_split, BASE_MAX_YEAR, CALIB_YEAR
from tune import SEARCH_SPACE, DEFAULT_PARAMS, TUNE_RESULTS

FIGS = PROJECT / 'outputs' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')

## 1. The search space

Each axis spans values that XGBoost docs list as the typical operating band; together they cover under-/over-fitting and the regularization-vs-signal trade-off without exploding the grid.

In [ ]:
grid_size = np.prod([len(v) for v in SEARCH_SPACE.values()])
print(f'Search axes: {len(SEARCH_SPACE)}')
print(f'Total grid points: {grid_size}')
print()
for k, v in SEARCH_SPACE.items():
    print(f'  {k:20s} {v}')
print(f'\nPhase-4 baseline (trial 0): {DEFAULT_PARAMS}')

## 2. Sweep results

In [ ]:
results = pd.read_csv(TUNE_RESULTS)
print(f'Loaded {len(results)} trials from {TUNE_RESULTS.name}')
results = results.sort_values('mean_auc', ascending=False).reset_index(drop=True)
results.round(4)

In [ ]:
# Where does the Phase-4 default rank?
default_rank = int(results.index[results['is_default']][0])
default_auc = float(results.loc[results['is_default'], 'mean_auc'].iloc[0])
best_auc = float(results['mean_auc'].iloc[0])
print(f'Phase-4 defaults rank: {default_rank + 1}/{len(results)}  (AUC={default_auc:.4f})')
print(f'Best trial AUC:        {best_auc:.4f}')
print(f'CV AUC headroom:       {best_auc - default_auc:+.4f}')

## 3. What did the search actually move?

If trials with higher `mean_auc` consistently sit at one end of a parameter's range, that's a signal the default was wrong on that axis. If the spread is random, that parameter doesn't matter much for this dataset.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for ax, param in zip(axes.flat, list(SEARCH_SPACE.keys())):
    ax.scatter(results[param], results['mean_auc'], alpha=0.7,
               c=['tomato' if d else 'steelblue' for d in results['is_default']])
    ax.set_xlabel(param)
    ax.set_ylabel('Mean CV ROC-AUC')
    ax.set_title(param)
fig.suptitle('Parameter → CV score (red = Phase-4 default)')
fig.tight_layout()
fig.savefig(FIGS / 'phase7_param_scatter.png', dpi=120)
plt.show()

## 4. Load `xgb_v3_tuned` and evaluate on test

The smoke already refit the best params on full base (2007-2015), calibrated on 2016, and saved the result. We just load it and re-run the Phase 5 evaluation surface.

In [ ]:
train, test = load_processed()
X_test, y_test = split_xy(test)

v2 = joblib.load(model_path('xgb_v2_calibrated'))
v3 = joblib.load(model_path('xgb_v3_tuned'))

p_v2 = v2.predict_proba(X_test)[:, 1]
p_v3 = v3.predict_proba(X_test)[:, 1]

m_v2 = evaluate(y_test, p_v2)
m_v3 = evaluate(y_test, p_v3)

pd.DataFrame({
    'xgb_v2_calibrated (Phase 5)': m_v2.as_row(),
    'xgb_v3_tuned (Phase 7)':      m_v3.as_row(),
}).T.round(4)

In [ ]:
# Delta table — same metrics, columns mean is improvement direction.
deltas = pd.DataFrame([
    {'metric': k,
     'v2': getattr(m_v2, k),
     'v3': getattr(m_v3, k),
     'delta': getattr(m_v3, k) - getattr(m_v2, k)}
    for k in ['roc_auc', 'pr_auc', 'brier', 'log_loss', 'ks']
])
deltas['better_for_v3'] = deltas.apply(
    lambda r: (r['delta'] < 0) if r['metric'] in ('brier', 'log_loss') else (r['delta'] > 0),
    axis=1,
)
deltas.round(4)

## 5. Gains/lift comparison

ROC-AUC and KS measure overall ranking. Operationally the question is sharper: **does v3 push more defaults into the top decile than v2?** That's where the cost-curve threshold actually operates.

In [ ]:
from evaluate import gains_table

gt_v2 = gains_table(y_test, p_v2, n_bins=10)
gt_v3 = gains_table(y_test, p_v3, n_bins=10)

lift_compare = pd.DataFrame({
    'decile': gt_v2.index,
    'lift_v2': gt_v2['lift'].values,
    'lift_v3': gt_v3['lift'].values,
    'cum_capture_v2': gt_v2['cum_capture_rate'].values,
    'cum_capture_v3': gt_v3['cum_capture_rate'].values,
}).set_index('decile')
lift_compare.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
x = np.arange(1, 11)
w = 0.35
axes[0].bar(x - w/2, lift_compare['lift_v2'], width=w, label='v2', color='steelblue')
axes[0].bar(x + w/2, lift_compare['lift_v3'], width=w, label='v3 (tuned)', color='tomato')
axes[0].axhline(1.0, color='grey', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Decile (1 = highest predicted risk)')
axes[0].set_ylabel('Lift over base default rate')
axes[0].set_title('Lift by decile')
axes[0].legend()

axes[1].plot(x, lift_compare['cum_capture_v2'], marker='o', label='v2', color='steelblue')
axes[1].plot(x, lift_compare['cum_capture_v3'], marker='s', label='v3 (tuned)', color='tomato')
axes[1].plot([1, 10], [0.1, 1.0], '--', color='grey', alpha=0.5, label='random')
axes[1].set_xlabel('Decile')
axes[1].set_ylabel('Cumulative % defaults captured')
axes[1].set_title('Cumulative gains')
axes[1].legend()

fig.tight_layout()
fig.savefig(FIGS / 'phase7_gains_compare.png', dpi=120)
plt.show()

## 6. Findings

1. **Phase-4 defaults ranked dead last (15/15)** under identical CV. CV ROC-AUC: default = 0.7008, best = 0.7239 — headroom **+0.0231**. The hand-picked starting point was the worst row in the sweep.
2. **What moved the needle: regularization.** Every top-5 trial shares `reg_lambda = 5.0` (vs default 1.0) and `gamma >= 0` with `min_child_weight in {5, 20}`. The top trial also used `max_depth = 4` (default was 6) and `n_estimators = 200` (default 400). Read: Phase 4 was *under-regularized* — deeper trees and weaker L2 were overfitting the training distribution. `learning_rate` and `subsample` looked roughly indifferent in the explored range.
3. **Test-set improvement, all five metrics in the right direction but small:**
   - ROC-AUC: 0.7000 → 0.7040  (+0.0040)
   - PR-AUC:  0.4405 → 0.4454  (+0.0049)
   - Brier:   0.1800 → 0.1788  (-0.0012)
   - KS:      0.2907 → 0.2975  (+0.0067)
   - Decile-1 lift: 1.9261× → 1.9428× (+0.0167)
4. **The CV gain (+0.023) did NOT fully transfer to test (+0.004 ROC-AUC).** That's the CV-test gap caused by concept drift — the held-out folds inside 2007-2015 share macro conditions with the training folds, while 2017-2018 has drifted further. The sweep correctly identified the *direction* of improvement (more regularization), but the absolute test ceiling is governed by drift the sweep can't see.
5. **Verdict: promote v3.** Every test metric improved, the model is also *smaller* (max_depth=4, n_estimators=200 → cheaper to fit and faster to predict), and there's no downside. Update `MODEL_NAME=xgb_v3_tuned` in `serve.py`'s env and the rest of the deployment stack picks it up.
6. **Where the next 5 KS points are.** Not in more hyperparameter trials — the test-set ceiling looks close. Better levers:
   - **Interaction features** (`int_rate × term`, `loan_amnt / annual_inc`, `fico_mean × dti`) — these are exactly the non-linear combinations a depth-4 tree can capture but only one at a time.
   - **Sub-grade-conditional scoring** — fit a small model *per grade band* and route by grade at score time.
   - **More recent calibration** — refresh the isotonic on rolling-12-month data once production starts collecting labels.